# 📚 내 기록 → 마크다운 변환기

흩어져 있는 내 지식·경험을 **AI가 읽기 좋은 마크다운(.md)** 으로 모읍니다.

> PKOS(개인지식운영체계) 프로젝트

### 이 노트북으로 할 수 있는 것

| | 무엇을 | 어떤 형식 |
|---|---|---|
| **1부** | 네이버 블로그 백업 | `.pdf` (글마다 나눠서 변환) |
| **2부** | 문서 폴더 통째로 | `.hwp` `.hwpx` `.docx` `.pptx` `.xlsx` `.pdf` `.html` `.txt` |
| **3부** | 구글 문서 | 구글 문서·시트·슬라이드 |

---

### 사용 방법

**먼저 아래 '준비하기' 두 칸을 실행**한 뒤, 필요한 부(1·2·3)로 가서
각 칸의 **▶ 버튼**을 순서대로 누르면 됩니다.

- 중간에 끊겨도 다시 누르면 **이어서** 진행됩니다
- 한 파일이 실패해도 나머지는 계속 변환됩니다
- 모든 작업은 **본인 구글 드라이브 안에서만** 이루어집니다

⏱️ 파일 100MB당 대략 1~3분.

## 🔧 준비하기 (설치 + 구글 드라이브 연결) — 맨 처음 한 번

▶ 를 누르면 구글 계정 접근 허용을 물어봅니다. **허용**을 눌러주세요.
내 드라이브 안에서만 작업하며, 파일이 외부로 나가지 않습니다.

In [ ]:
#@title ▶ 눌러서 준비하기 { display-mode: "form" }
import subprocess, sys

print("① 필요한 프로그램 설치 중... (30초쯤 걸립니다)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pymupdf",          # PDF
                "olefile",          # 한글 .hwp
                "python-docx",      # 워드
                "python-pptx",      # 파워포인트
                "openpyxl",         # 엑셀
                "beautifulsoup4",   # HTML
                ], check=False)

print("② 구글 드라이브 연결 중...")
from google.colab import drive
drive.mount('/content/drive')

print("\n준비 완료! 다음 칸으로 넘어가세요.")

### (자동) 변환 엔진 불러오기 — 이 칸도 ▶ 눌러주세요

In [ ]:
#@title ▶ 눌러서 엔진 불러오기 { display-mode: "form" }
import base64, pathlib, importlib, sys

_ENGINES = {
  "pkos_paths.py": (
    "IiIi7L2U656pIOuwjyBXaW5kb3dzIEdvb2dsZSBEcml2ZSDqsr3roZzrpbwg64K0IOuTnOudvOydtOu4jCDqsr3roZzroZwg7KCV"
    "6rec7ZmU7ZWc64ukLiIiIgppbXBvcnQgcG9zaXhwYXRoCmltcG9ydCByZQoKTVlEUklWRSA9ICIvY29udGVudC9kcml2ZS9NeURy"
    "aXZlIgoKCmRlZiBkcml2ZV9wYXRoKHZhbHVlOiBzdHIpIC0+IHN0cjoKICAgIHRleHQgPSAodmFsdWUgb3IgIiIpLnN0cmlwKCku"
    "c3RyaXAoJyJcJ+KAnOKAneKAmOKAmScpLnN0cmlwKCkucmVwbGFjZSgiXFwiLCAiLyIpCiAgICBpZiBub3QgdGV4dDoKICAgICAg"
    "ICByYWlzZSBWYWx1ZUVycm9yKCLtj7TrjZQg6rK966Gc66W8IOyeheugpe2VtOyjvOyEuOyalC4g7L2U656p7JeQ7IScICfqsr3r"
    "oZwg67O17IKsJ+2VnCDqsJLsnYQg67aZ7Jes64Sj7Jy87IS47JqULiIpCiAgICBpZiByZS5tYXRjaChyImh0dHBzPzovLyIsIHRl"
    "eHQpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuydtCDsubjsnYAg7Y+0642UIOunge2BrOuCmCBJROqwgCDslYTri4wg6rK9"
    "66Gc66W8IOuwm+yKteuLiOuLpC4g7Jm87Kq9IO2MjOydvCDtg5Dsg4nquLDsl5DshJwg7Y+0642U7J2YICfqsr3roZwg67O17IKs"
    "J+ulvCDsgqzsmqntlZjshLjsmpQuIikKICAgIHdpbmRvd3MgPSByZS5tYXRjaChyIl5bQS1aYS16XTovKD8664K0IOuTnOudvOyd"
    "tOu4jHxNeSBEcml2ZXxNeURyaXZlKSg/Oi98JCkiLCB0ZXh0KQogICAgaWYgd2luZG93czoKICAgICAgICB0ZXh0ID0gdGV4dFt3"
    "aW5kb3dzLmVuZCgpOl0KICAgIGVsaWYgcmUubWF0Y2gociJeW0EtWmEtel06IiwgdGV4dCk6CiAgICAgICAgcmFpc2UgVmFsdWVF"
    "cnJvcigi64K0IOuTnOudvOydtOu4jCDqsr3roZzrp4wg7KeA7JuQ7ZWp64uI64ukLiDsvZTrnqnsl5DshJwg7Y+0642U7J2YIOqy"
    "veuhnOulvCDrs7XsgqztlbTso7zshLjsmpQuIikKICAgIGVsaWYgdGV4dCA9PSBNWURSSVZFIG9yIHRleHQuc3RhcnRzd2l0aChN"
    "WURSSVZFICsgIi8iKToKICAgICAgICB0ZXh0ID0gdGV4dFtsZW4oTVlEUklWRSk6XS5sc3RyaXAoIi8iKQogICAgZWxpZiB0ZXh0"
    "LnN0YXJ0c3dpdGgoIi8iKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCIvY29udGVudC9kcml2ZS9NeURyaXZlIOyViOydmCDq"
    "sr3roZzrpbwg7J6F66Cl7ZW07KO87IS47JqULiIpCiAgICByZXN1bHQgPSBwb3NpeHBhdGgubm9ybXBhdGgocG9zaXhwYXRoLmpv"
    "aW4oTVlEUklWRSwgdGV4dCkpCiAgICBpZiByZXN1bHQgIT0gTVlEUklWRSBhbmQgbm90IHJlc3VsdC5zdGFydHN3aXRoKE1ZRFJJ"
    "VkUgKyAiLyIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIuuCtCDrk5zrnbzsnbTruIwg67CU6rml7J2YIOqyveuhnOuKlCDs"
    "gqzsmqntlaAg7IiYIOyXhuyKteuLiOuLpC4iKQogICAgcmV0dXJuIHJlc3VsdAo="
  ),
  "pkos_converter.py": (
    "..."
    "ZSwgInBvc3RzIjogbGVuKGZvdW5kKX0K"
  ),
  "pkos_readers.py": (
    "..."
  ),
  "pkos_privacy.py": (
    "..."
    "ICAgcmV0dXJuIGhpdHMK"
  ),
  "pkos_folder.py": (
    "IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiClBLT1Mg7Y+0642UIOydvOq0hCDrs4DtmZjquLAKPT09PT09PT09PT09PT09PT09"
    "PT09PT0K7Y+0642UIO2VmOuCmOulvCDthrXsp7jroZwg7ZuR7Ja07IScLCDslYjsl5Ag7J6I64qUIOuqqOuToCDrrLjshJzrpbwg"
    "66eI7YGs64uk7Jq0KC5tZCnsnLzroZwg67CU6r6864ukLgrtlZzquIAoLmh3cC8uaHdweCksIOybjOuTnCwg7YyM7JuM7Y+s7J24"
    "7Yq4LCDsl5HshYAsIFBERiwgSFRNTCwg7YWN7Iqk7Yq466W8IOuqqOuRkCDri6Tro6zri6QuCgogICAgZnJvbSBwa29zX2ZvbGRl"
    "ciBpbXBvcnQgRm9sZGVyQ29udmVydGVyLCBGb2xkZXJTZXR0aW5ncwoKICAgIGZjID0gRm9sZGVyQ29udmVydGVyKEZvbGRlclNl"
    "dHRpbmdzKAogICAgICAgIHNyY19kaXIgPSAiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS8wMV/tlZnqtZAiLAogICAgICAgIG91dF9k"
    "aXIgPSAiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9QS09TL+uzgO2ZmOqysOqzvCIsCiAgICApKQogICAgZmMuc2NhbigpICAgICAg"
    "IyDrqLzsoIAg66y07JeH7J20IOuqhyDqsJwg7J6I64qU7KeAIO2ZleyduAogICAgZmMucnVuKCkgICAgICAgIyDrs4DtmZgKCu2K"
    "ueynlQogICAgLSDsm5Drnpgg7Y+0642UIOq1rOyhsOulvCDqt7jrjIDroZwg7Jyg7KeA7ZWc64ukCiAgICAtIOydtOuvuCDrs4Dt"
    "mZjtlZwg7YyM7J287J2AIOqxtOuEiOubtOuLpCAo7KSR6rCE7JeQIOuBiuqyqOuPhCDsnbTslrTshJwg7KeE7ZaJKQogICAgLSDt"
    "lZwg7YyM7J287J20IOyLpO2MqO2VtOuPhCDsoITssrTqsIAg66mI7LaU7KeAIOyViuuKlOuLpCAo7Jik66WY64qUIOuUsOuhnCDq"
    "uLDroZ0pCiAgICAtIOuzgO2ZmCDqsrDqs7wg66qp66GdKElOREVYLm1kLCBfZmlsZXMuanNvbinsnYQg66eM65Og64ukCgpQS09T"
    "KOqwnOyduOyngOyLneyatOyYgeyytOqzhCkg7ZSE66Gc7KCd7Yq4CiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0"
    "aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBpbwppbXBvcnQgcmUKaW1wb3J0IGpzb24KaW1wb3J0IHRpbWUKaW1wb3J0IGNvbGxlY3Rp"
    "b25zCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKCmZyb20gcGtvc19yZWFkZXJzIGltcG9ydCByZWFk"
    "X2FueSwgUkVBREVSUywgUmVhZFJlc3VsdApmcm9tIHBrb3NfcHJpdmFjeSBpbXBvcnQgUHJpdmFjeUZpbHRlciwgUG9saWN5LCBQ"
    "cml2YWN5UmVwb3J0CgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKQGRhdGFjbGFzcwpjbGFzcyBGb2xkZXJT"
    "ZXR0aW5nczoKICAgIHNyY19kaXI6IHN0ciAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyDtm5HsnYQg7JuQ67O4"
    "IO2PtOuNlAogICAgb3V0X2Rpcjogc3RyID0gIiIgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIOqysOqzvCDtj7TrjZQg"
    "KOu5hOyasOuptCBzcmNfZGlyL19tZCkKICAgIGluY2x1ZGU6IHR1cGxlID0gdHVwbGUoUkVBREVSUykgICAgICAgICAgICAgICAg"
    "IyDri6Tro7Ag7ZmV7J6l7J6QCiAgICBleGNsdWRlX2RpcnM6IHR1cGxlID0gKCIuZ2l0IiwgIi5vYnNpZGlhbiIsICJfX3B5Y2Fj"
    "aGVfXyIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICJub2RlX21vZHVsZXMiLCAiaW1hZ2VzIiwgIl9tZCIpCiAgICBza2lw"
    "X2V4aXN0aW5nOiBib29sID0gVHJ1ZSAgICAgICAgICAgICAgICAgICAgICMg7J2066+4IOyeiOuKlCBtZCDripQg6rG064SI65uw"
    "6riwCiAgICBtaW5fY2hhcnM6IGludCA9IDEwICAgICAgICAgICAgICAgICAgICAgICAgICAgICMg7J2067O064ukIOynp+ycvOup"
    "tCAn64K07JqpIOyXhuydjCfsnLzroZwg6riw66GdCiAgICBtYXhfbWI6IGZsb2F0ID0gMjAwLjAgICAgICAgICAgICAgICAgICAg"
    "ICAgICAgICMg7J2067O064ukIO2BsCDtjIzsnbzsnYAg6rG064SI65uw6riwCiAgICBrZWVwX3RyZWU6IGJvb2wgPSBUcnVlICAg"
    "ICAgICAgICAgICAgICAgICAgICAgICMg7JuQ67O4IO2PtOuNlCDqtazsobAg7Jyg7KeACiAgICB2ZXJib3NlOiBib29sID0gVHJ1"
    "ZQoKICAgICMg4pSA4pSAIOqwnOyduOygleuztCDsspjrpqwg4pSA4pSACiAgICDqsJzsnbjsoJXrs7Rf6rCA66as6riwOiBib29s"
    "ID0gVHJ1ZSAgICAgICAgICAgICAgICAgICAgIyDrgYTrqbQg7JuQ66y4IOq3uOuMgOuhnCDsoIDsnqUKICAgIOqwnOyduOygleuz"
    "tF/soJXssYU6IFBvbGljeSA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1Qb2xpY3kpCiAgICDrs7Tqs6DshJxf7JuQ67O47ZGc7Iuc"
    "OiBib29sID0gVHJ1ZSAgICAgICAgICAgICAgICAgICAgIyDrs7Tqs6DshJzsl5Ag6rCA66as6riwIOyghCDqsJLsnYQg64Ko6ri4"
    "7KeACgogICAgZGVmIHJlc29sdmVkX291dChzZWxmKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIHNlbGYub3V0X2RpciBvciBvcy5w"
    "YXRoLmpvaW4oc2VsZi5zcmNfZGlyLCAiX21kIikKCgpfSU5WQUxJRCA9IHJlLmNvbXBpbGUocidbXFwvOio/Ijw+fF0nKQoKCmRl"
    "ZiBzYWZlX25hbWUobmFtZTogc3RyLCBtYXhsZW46IGludCA9IDkwKSAtPiBzdHI6CiAgICBzID0gX0lOVkFMSUQuc3ViKCJfIiwg"
    "bmFtZSkuc3RyaXAoKQogICAgcmV0dXJuIHNbOm1heGxlbl0ucnN0cmlwKCIgLiIpIG9yICLrrLTsoJwiCgoKIyDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi"
    "lIDilIDilIDilIDilIDilIDilIDilIAKY2xhc3MgRm9sZGVyQ29udmVydGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHNldHRp"
    "bmdzOiBGb2xkZXJTZXR0aW5ncyk6CiAgICAgICAgc2VsZi5zID0gc2V0dGluZ3MKICAgICAgICBzZWxmLm91dCA9IHNldHRpbmdz"
    "LnJlc29sdmVkX291dCgpCiAgICAgICAgc2VsZi5yZWNvcmRzOiBsaXN0W2RpY3RdID0gW10KICAgICAgICBzZWxmLmVycm9yczog"
    "bGlzdFtkaWN0XSA9IFtdCiAgICAgICAgc2VsZi5zdGF0cyA9IGNvbGxlY3Rpb25zLkNvdW50ZXIoKQogICAgICAgIHNlbGYucHJp"
    "dmFjeSA9IFByaXZhY3lGaWx0ZXIoc2V0dGluZ3Mu6rCc7J247KCV67O0X+ygleyxhSkgXAogICAgICAgICAgICBpZiBzZXR0aW5n"
    "cy7qsJzsnbjsoJXrs7Rf6rCA66as6riwIGVsc2UgTm9uZQogICAgICAgIHNlbGYucmVwb3J0ID0gUHJpdmFjeVJlcG9ydChzaG93"
    "X29yaWdpbmFsPXNldHRpbmdzLuuztOqzoOyEnF/sm5Drs7jtkZzsi5wpCgogICAgZGVmIGxvZyhzZWxmLCAqYSk6CiAgICAgICAg"
    "aWYgc2VsZi5zLnZlcmJvc2U6CiAgICAgICAgICAgIHByaW50KCphLCBmbHVzaD1UcnVlKQoKICAgICMg4pSA4pSAIOuMgOyDgSDt"
    "jIzsnbwg7IiY7KeRCiAgICBkZWYgY29sbGVjdChzZWxmKSAtPiBsaXN0W3N0cl06CiAgICAgICAgZm91bmQgPSBbXQogICAgICAg"
    "IGV4dHMgPSB7ZS5sb3dlcigpIGZvciBlIGluIHNlbGYucy5pbmNsdWRlfQogICAgICAgIG91dF9hYnMgPSBvcy5wYXRoLmFic3Bh"
    "dGgoc2VsZi5vdXQpCiAgICAgICAgZm9yIGRwLCBkbnMsIGZucyBpbiBvcy53YWxrKHNlbGYucy5zcmNfZGlyKToKICAgICAgICAg"
    "ICAgZG5zWzpdID0gW2QgZm9yIGQgaW4gZG5zCiAgICAgICAgICAgICAgICAgICAgICBpZiBkIG5vdCBpbiBzZWxmLnMuZXhjbHVk"
    "ZV9kaXJzIGFuZCBub3QgZC5zdGFydHN3aXRoKCIuIildCiAgICAgICAgICAgIGlmIG9zLnBhdGguYWJzcGF0aChkcCkuc3RhcnRz"
    "d2l0aChvdXRfYWJzKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAj"
    "IOqysOqzvCDtj7TrjZTripQg7KCc7Jm4CiAgICAgICAgICAgIGZvciBmbiBpbiBmbnM6CiAgICAgICAgICAgICAgICBpZiBmbi5z"
    "dGFydHN3aXRoKCJ+JCIpIG9yIGZuLnN0YXJ0c3dpdGgoIi4iKToKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAg"
    "ICAgICAgICAgaWYgb3MucGF0aC5zcGxpdGV4dChmbilbMV0ubG93ZXIoKSBpbiBleHRzOgogICAgICAgICAgICAgICAgICAgIGZv"
    "dW5kLmFwcGVuZChvcy5wYXRoLmpvaW4oZHAsIGZuKSkKICAgICAgICByZXR1cm4gc29ydGVkKGZvdW5kKQoKICAgICMg4pSA4pSA"
    "IO2bkeyWtOuztOq4sCAo67OA7ZmYIOyXhuydtCDtmITtmanrp4wpCiAgICBkZWYgc2NhbihzZWxmKSAtPiBkaWN0OgogICAgICAg"
    "IGZpbGVzID0gc2VsZi5jb2xsZWN0KCkKICAgICAgICBieV9leHQgPSBjb2xsZWN0aW9ucy5Db3VudGVyKG9zLnBhdGguc3BsaXRl"
    "eHQoZilbMV0ubG93ZXIoKSBmb3IgZiBpbiBmaWxlcykKICAgICAgICB0b3RhbF9tYiA9IDAuMAogICAgICAgIGZvciBmIGluIGZp"
    "bGVzOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB0b3RhbF9tYiArPSBvcy5wYXRoLmdldHNpemUoZikgLyAxMDI0"
    "IC8gMTAyNAogICAgICAgICAgICBleGNlcHQgT1NFcnJvcjoKICAgICAgICAgICAgICAgIHBhc3MKCiAgICAgICAgc2VsZi5sb2co"
    "ZiLrjIDsg4Eg7Y+0642UIDoge3NlbGYucy5zcmNfZGlyfSIpCiAgICAgICAgc2VsZi5sb2coZiLssL7snYAg7YyM7J28IDoge2xl"
    "bihmaWxlcyl96rCcICh7dG90YWxfbWI6LjBmfSBNQilcbiIpCiAgICAgICAgc2VsZi5sb2coIiAg7ZiV7Iud67OEIikKICAgICAg"
    "ICBmb3IgZSwgbiBpbiBieV9leHQubW9zdF9jb21tb24oKToKICAgICAgICAgICAgc2VsZi5sb2coZiIgICAge2U6OH0ge246NX3q"
    "sJwiKQogICAgICAgIHNlbGYubG9nKGYiXG4gIOyggOyepSDsnITsuZggOiB7c2VsZi5vdXR9IikKICAgICAgICByZXR1cm4geyJm"
    "aWxlcyI6IGxlbihmaWxlcyksICJieV9leHQiOiBkaWN0KGJ5X2V4dCksICJtYiI6IHJvdW5kKHRvdGFsX21iKX0KCiAgICAjIOKU"
    "gOKUgCDqsrDqs7wgbWQg6rK966GcIOygle2VmOq4sAogICAgZGVmIG1kX3BhdGhfZm9yKHNlbGYsIHNyYzogc3RyKSAtPiBzdHI6"
    "CiAgICAgICAgcmVsID0gb3MucGF0aC5yZWxwYXRoKHNyYywgc2VsZi5zLnNyY19kaXIpCiAgICAgICAgaGVhZCwgZm4gPSBvcy5w"
    "YXRoLnNwbGl0KHJlbCkKICAgICAgICBzdGVtLCBleHQgPSBvcy5wYXRoLnNwbGl0ZXh0KGZuKQogICAgICAgIG5hbWUgPSBzYWZl"
    "X25hbWUoZiJ7c3RlbX17ZXh0LnJlcGxhY2UoJy4nLCAnXycpfSIpICsgIi5tZCIKICAgICAgICBpZiBzZWxmLnMua2VlcF90cmVl"
    "IGFuZCBoZWFkIGFuZCBoZWFkICE9ICIuIjoKICAgICAgICAgICAgaGVhZCA9IG9zLnBhdGguam9pbigqW3NhZmVfbmFtZShwKSBm"
    "b3IgcCBpbiBoZWFkLnNwbGl0KG9zLnNlcCldKQogICAgICAgICAgICByZXR1cm4gb3MucGF0aC5qb2luKHNlbGYub3V0LCBoZWFk"
    "LCBuYW1lKQogICAgICAgIHJldHVybiBvcy5wYXRoLmpvaW4oc2VsZi5vdXQsIG5hbWUpCgogICAgIyDilIDilIAg66i466as66eQ"
    "IOunjOuTpOq4sAogICAgZGVmIGZyb250X21hdHRlcihzZWxmLCBzcmM6IHN0ciwgcmVzOiBSZWFkUmVzdWx0KSAtPiBzdHI6CiAg"
    "ICAgICAgc3QgPSBvcy5zdGF0KHNyYykKICAgICAgICBtdGltZSA9IHRpbWUuc3RyZnRpbWUoIiVZLSVtLSVkICVIOiVNIiwgdGlt"
    "ZS5sb2NhbHRpbWUoc3Quc3RfbXRpbWUpKQogICAgICAgIHJlbCA9IG9zLnBhdGgucmVscGF0aChzcmMsIHNlbGYucy5zcmNfZGly"
    "KS5yZXBsYWNlKCJcXCIsICIvIikKICAgICAgICB0aXRsZSA9IG9zLnBhdGguc3BsaXRleHQob3MucGF0aC5iYXNlbmFtZShzcmMp"
    "KVswXS5yZXBsYWNlKCciJywgIiciKQogICAgICAgIGV4dHJhID0gIiIKICAgICAgICBpZiByZXMubWV0YToKICAgICAgICAgICAg"
    "Yml0cyA9ICIgwrcgIi5qb2luKGYie2t9IHt2fSIgZm9yIGssIHYgaW4gcmVzLm1ldGEuaXRlbXMoKQogICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAgICBpZiBrIG5vdCBpbiAoInVybCIsICJkb2NfaWQiKSkKICAgICAgICAgICAgaWYgYml0czoKICAgICAgICAg"
    "ICAgICAgIGV4dHJhID0gZiJpbmZvOiB7Yml0c31cbiIKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICAiLS0tXG4iCiAgICAg"
    "ICAgICAgIGYndGl0bGU6ICJ7dGl0bGV9IlxuJwogICAgICAgICAgICBmImRhdGU6IHttdGltZX1cbiIKICAgICAgICAgICAgZidz"
    "b3VyY2U6ICJ7cmVsfSJcbicKICAgICAgICAgICAgZidraW5kOiAie3Jlcy5raW5kfSJcbicKICAgICAgICAgICAgZiJ7ZXh0cmF9"
    "IgogICAgICAgICAgICAiLS0tXG5cbiIKICAgICAgICAgICAgZiIjIHt0aXRsZX1cblxuIgogICAgICAgICAgICBmIirsm5Drs7g6"
    "IGB7cmVsfWAgwrcg7IiY7KCVIHttdGltZX0qXG5cbiIKICAgICAgICApCgogICAgIyDilIDilIAg7Iuk7ZaJCiAgICBkZWYgcnVu"
    "KHNlbGYsIGxpbWl0OiBpbnQgfCBOb25lID0gTm9uZSkgLT4gZGljdDoKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAg"
    "b3MubWFrZWRpcnMoc2VsZi5vdXQsIGV4aXN0X29rPVRydWUpCiAgICAgICAgZmlsZXMgPSBzZWxmLmNvbGxlY3QoKQogICAgICAg"
    "IGlmIGxpbWl0OgogICAgICAgICAgICBmaWxlcyA9IGZpbGVzWzpsaW1pdF0KICAgICAgICB0b3RhbCA9IGxlbihmaWxlcykKICAg"
    "ICAgICBzZWxmLmxvZyhmIu2MjOydvCB7dG90YWx96rCc66W8IOuzgO2ZmO2VqeuLiOuLpC5cbiIpCgogICAgICAgIGRvbmUgPSBz"
    "a2lwcGVkID0gZmFpbGVkID0gMAogICAgICAgIGZvciBpLCBzcmMgaW4gZW51bWVyYXRlKGZpbGVzLCAxKToKICAgICAgICAgICAg"
    "ZHN0ID0gc2VsZi5tZF9wYXRoX2ZvcihzcmMpCgogICAgICAgICAgICBpZiBzZWxmLnMuc2tpcF9leGlzdGluZyBhbmQgb3MucGF0"
    "aC5leGlzdHMoZHN0KToKICAgICAgICAgICAgICAgIHNraXBwZWQgKz0gMQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAg"
    "ICAgICAgdHJ5OgogICAgICAgICAgICAgICAgbWIgPSBvcy5wYXRoLmdldHNpemUoc3JjKSAvIDEwMjQgLyAxMDI0CiAgICAgICAg"
    "ICAgIGV4Y2VwdCBPU0Vycm9yOgogICAgICAgICAgICAgICAgbWIgPSAwCiAgICAgICAgICAgIGlmIG1iID4gc2VsZi5zLm1heF9t"
    "YjoKICAgICAgICAgICAgICAgIHNlbGYuZXJyb3JzLmFwcGVuZCh7ImZpbGUiOiBzcmMsICJlcnJvciI6IGYi64SI66y0IO2BvCAo"
    "e21iOi4wZn1NQikifSkKICAgICAgICAgICAgICAgIGZhaWxlZCArPSAxCiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAg"
    "ICAgICAgcmVzID0gcmVhZF9hbnkoc3JjKQogICAgICAgICAgICBpZiBub3QgcmVzLm9rOgogICAgICAgICAgICAgICAgc2VsZi5l"
    "cnJvcnMuYXBwZW5kKHsiZmlsZSI6IHNyYywgImVycm9yIjogcmVzLmVycm9yfSkKICAgICAgICAgICAgICAgIHNlbGYuc3RhdHNb"
    "ZiLsi6TtjKg6e3Jlcy5raW5kfSJdICs9IDEKICAgICAgICAgICAgICAgIGZhaWxlZCArPSAxCiAgICAgICAgICAgICAgICBjb250"
    "aW51ZQoKICAgICAgICAgICAgYm9keSA9IHJlcy50ZXh0CiAgICAgICAgICAgIHJlbCA9IG9zLnBhdGgucmVscGF0aChzcmMsIHNl"
    "bGYucy5zcmNfZGlyKS5yZXBsYWNlKCJcXCIsICIvIikKCiAgICAgICAgICAgIOqwnOyduOygleuztF/qsbTsiJggPSAwCiAgICAg"
    "ICAgICAgIGlmIHNlbGYucHJpdmFjeSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGJvZHksIGhpdHMgPSBzZWxmLnByaXZh"
    "Y3kubWFzayhib2R5KQogICAgICAgICAgICAgICAgaWYgaGl0czoKICAgICAgICAgICAgICAgICAgICBzZWxmLnJlcG9ydC5hZGQo"
    "cmVsLCBoaXRzKQogICAgICAgICAgICAgICAgICAgIOqwnOyduOygleuztF/qsbTsiJggPSBsZW4oaGl0cykKICAgICAgICAgICAg"
    "ICAgICAgICBzZWxmLnN0YXRzWyLqsJzsnbjsoJXrs7TqsIDrprwiXSArPSBsZW4oaGl0cykKCiAgICAgICAgICAgIG5vdGUgPSAi"
    "IgogICAgICAgICAgICBpZiBsZW4oYm9keSkgPCBzZWxmLnMubWluX2NoYXJzOgogICAgICAgICAgICAgICAgbm90ZSA9ICgiXG4+"
    "IOKaoCDquIDsnpDrpbwg6rGw7J2YIOywvuyngCDrqrvtlojsirXri4jri6QuIOq3uOumvCDsnITso7zsnbTqsbDrgpgg7Iqk7LqU"
    "7ZWcIOusuOyEnOydvCDsiJggIgogICAgICAgICAgICAgICAgICAgICAgICAi7J6I7Iq164uI64ukLiDsnbQg64+E6rWs64qUIOq4"
    "gOyekCDsnbjsi50oT0NSKeydhCDtlZjsp4Ag7JWK7Iq164uI64ukLlxuIikKICAgICAgICAgICAgICAgIHNlbGYuc3RhdHNbIuuC"
    "tOyaqeqxsOydmOyXhuydjCJdICs9IDEKCiAgICAgICAgICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShkc3QpLCBleGlz"
    "dF9vaz1UcnVlKQogICAgICAgICAgICB3aXRoIGlvLm9wZW4oZHN0LCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAg"
    "ICAgICAgICAgICBmLndyaXRlKHNlbGYuZnJvbnRfbWF0dGVyKHNyYywgcmVzKSArIG5vdGUgKyBib2R5ICsgIlxuIikKCiAgICAg"
    "ICAgICAgIHNlbGYucmVjb3Jkcy5hcHBlbmQoewogICAgICAgICAgICAgICAgInRpdGxlIjogb3MucGF0aC5zcGxpdGV4dChvcy5w"
    "YXRoLmJhc2VuYW1lKHNyYykpWzBdLAogICAgICAgICAgICAgICAgImtpbmQiOiByZXMua2luZCwKICAgICAgICAgICAgICAgICJj"
    "aGFycyI6IHJlcy5jaGFycywKICAgICAgICAgICAgICAgICJzcmMiOiByZWwsCiAgICAgICAgICAgICAgICAibWQiOiBvcy5wYXRo"
    "LnJlbHBhdGgoZHN0LCBzZWxmLm91dCkucmVwbGFjZSgiXFwiLCAiLyIpLAogICAgICAgICAgICAgICAgImRhdGUiOiB0aW1lLnN0"
    "cmZ0aW1lKCIlWS0lbS0lZCIsIHRpbWUubG9jYWx0aW1lKG9zLnN0YXQoc3JjKS5zdF9tdGltZSkpLAogICAgICAgICAgICAgICAg"
    "IuqwnOyduOygleuztCI6IOqwnOyduOygleuztF/qsbTsiJgsCiAgICAgICAgICAgIH0pCiAgICAgICAgICAgIHNlbGYuc3RhdHNb"
    "cmVzLmtpbmRdICs9IDEKICAgICAgICAgICAgZG9uZSArPSAxCgogICAgICAgICAgICBpZiBkb25lIGFuZCBkb25lICUgNTAgPT0g"
    "MDoKICAgICAgICAgICAgICAgIHNlbGYubG9nKGYiICDigKYge2RvbmV96rCcIOuzgO2ZmCAoe2l9L3t0b3RhbH0pIikKCiAgICAg"
    "ICAgc2VsZi53cml0ZV9pbmRleCgpCiAgICAgICAg67O06rOg7IScID0gc2VsZi5yZXBvcnQud3JpdGUoc2VsZi5vdXQpIGlmIHNl"
    "bGYucHJpdmFjeSBpcyBub3QgTm9uZSBlbHNlIE5vbmUKCiAgICAgICAgc2VjcyA9IGludCh0aW1lLnRpbWUoKSAtIHQwKQogICAg"
    "ICAgIHNlbGYubG9nKGYiXG7smYTro4whIOuzgO2ZmCB7ZG9uZX3qsJwgwrcg6rG064SI65yAIHtza2lwcGVkfeqwnCDCtyDsi6Tt"
    "jKgge2ZhaWxlZH3qsJwgIgogICAgICAgICAgICAgICAgIGYiwrcge3NlY3MgLy8gNjB967aEIHtzZWNzICUgNjB97LSIIikKICAg"
    "ICAgICBpZiBzZWxmLnN0YXRzOgogICAgICAgICAgICBzZWxmLmxvZygiXG4gIO2YleyLneuzhCDqsrDqs7wiKQogICAgICAgICAg"
    "ICBmb3IgaywgbiBpbiBzZWxmLnN0YXRzLm1vc3RfY29tbW9uKCk6CiAgICAgICAgICAgICAgICBzZWxmLmxvZyhmIiAgICB7azox"
    "NH0ge246NX3qsJwiKQogICAgICAgIGlmIHNlbGYuZXJyb3JzOgogICAgICAgICAgICBzZWxmLmxvZyhmIlxuICDsi6TtjKgg66qp"
    "66GdIDoge29zLnBhdGguam9pbihzZWxmLm91dCwgJ1/smKTrpZgubWQnKX0iKQoKICAgICAgICBpZiBzZWxmLnByaXZhY3kgaXMg"
    "bm90IE5vbmU6CiAgICAgICAgICAgIHNlbGYubG9nKCJcbiIgKyAi4pSAIiAqIDQ2KQogICAgICAgICAgICBzZWxmLmxvZyhzZWxm"
    "LnJlcG9ydC5zdW1tYXJ5KCkpCiAgICAgICAgICAgIGlmIOuztOqzoOyEnDoKICAgICAgICAgICAgICAgIHNlbGYubG9nKGYiXG4g"
    "IOyekOyEuO2VnCDrgrTsl60gOiB767O06rOg7IScfSIpCiAgICAgICAgICAgICAgICBpZiBzZWxmLnMu67O06rOg7IScX+ybkOuz"
    "uO2RnOyLnDoKICAgICAgICAgICAgICAgICAgICBzZWxmLmxvZygiICDimqAg7J20IOuztOqzoOyEnOyXkOuKlCDqsIDrpqzquLAg"
    "7KCEIOybkOuzuOydtCDrk6TslrQg7J6I7Iq164uI64ukLiDqs7XsnKDtlZjsp4Ag66eI7IS47JqULiIpCgogICAgICAgIHJldHVy"
    "biB7ImRvbmUiOiBkb25lLCAic2tpcHBlZCI6IHNraXBwZWQsICJmYWlsZWQiOiBmYWlsZWQsCiAgICAgICAgICAgICAgICAic3Rh"
    "dHMiOiBkaWN0KHNlbGYuc3RhdHMpLAogICAgICAgICAgICAgICAgIuqwnOyduOygleuztCI6IGxlbihzZWxmLnJlcG9ydC5yb3dz"
    "KX0KCiAgICAjIOKUgOKUgCDrqqnssKgv7Jik66WYIOq4sOuhnQogICAgZGVmIHdyaXRlX2luZGV4KHNlbGYpOgogICAgICAgIGlm"
    "IHNlbGYucmVjb3JkczoKICAgICAgICAgICAgd2l0aCBpby5vcGVuKG9zLnBhdGguam9pbihzZWxmLm91dCwgIl9maWxlcy5qc29u"
    "IiksICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgICAgIGpzb24uZHVtcChzZWxmLnJlY29yZHMsIGYs"
    "IGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTEpCgogICAgICAgICAgICBieV9raW5kID0gY29sbGVjdGlvbnMuZGVmYXVsdGRp"
    "Y3QobGlzdCkKICAgICAgICAgICAgZm9yIHIgaW4gc2VsZi5yZWNvcmRzOgogICAgICAgICAgICAgICAgYnlfa2luZFtyWyJraW5k"
    "Il1dLmFwcGVuZChyKQoKICAgICAgICAgICAgTCA9IFsiIyDwn5OCIOuzgO2ZmOuQnCDrrLjshJwg66qp7LCoIiwgIiIsCiAgICAg"
    "ICAgICAgICAgICAgZiLsoITssrQgKip7bGVuKHNlbGYucmVjb3Jkcyl96rCcKiogwrcg7J6Q64+ZIOyDneyEsSIsICIiLAogICAg"
    "ICAgICAgICAgICAgICJ8IO2YleyLnSB8IOqwnOyImCB8IiwgInwtLS0tLS18LS0tLS06fCJdCiAgICAgICAgICAgIGZvciBrIGlu"
    "IHNvcnRlZChieV9raW5kLCBrZXk9bGFtYmRhIHg6IC1sZW4oYnlfa2luZFt4XSkpOgogICAgICAgICAgICAgICAgTC5hcHBlbmQo"
    "ZiJ8IHtrfSB8IHtsZW4oYnlfa2luZFtrXSl9IHwiKQogICAgICAgICAgICBMLmFwcGVuZCgiIikKICAgICAgICAgICAgZm9yIGsg"
    "aW4gc29ydGVkKGJ5X2tpbmQsIGtleT1sYW1iZGEgeDogLWxlbihieV9raW5kW3hdKSk6CiAgICAgICAgICAgICAgICBMICs9IFtm"
    "IiMjIHtrfSAoe2xlbihieV9raW5kW2tdKX3qsJwpIiwgIiJdCiAgICAgICAgICAgICAgICBmb3IgciBpbiBzb3J0ZWQoYnlfa2lu"
    "ZFtrXSwga2V5PWxhbWJkYSB4OiB4WyJzcmMiXSk6CiAgICAgICAgICAgICAgICAgICAgTC5hcHBlbmQoZiItIFt7clsndGl0bGUn"
    "XX1dKHtyWydtZCddfSkgIgogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYiwrcge3JbJ2NoYXJzJ106LH3snpAgwrcgYHty"
    "WydzcmMnXX1gIikKICAgICAgICAgICAgICAgIEwuYXBwZW5kKCIiKQogICAgICAgICAgICB3aXRoIGlvLm9wZW4ob3MucGF0aC5q"
    "b2luKHNlbGYub3V0LCAiSU5ERVgubWQiKSwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICAgICAgZi53"
    "cml0ZSgiXG4iLmpvaW4oTCkpCgogICAgICAgIGlmIHNlbGYuZXJyb3JzOgogICAgICAgICAgICBMID0gWyIjIOKaoCDrs4DtmZjt"
    "lZjsp4Ag66q77ZWcIO2MjOydvCIsICIiLAogICAgICAgICAgICAgICAgIGYie2xlbihzZWxmLmVycm9ycyl96rCcIiwgIiIsCiAg"
    "ICAgICAgICAgICAgICAgInwg7YyM7J28IHwg7J207Jyg7JmAIO2VtOqysCDrsKnrspUgfCIsICJ8LS0tLS0tfC0tLS0tLS0tLS0t"
    "LS0tLS0tLXwiXQogICAgICAgICAgICBmb3IgZSBpbiBzZWxmLmVycm9yczoKICAgICAgICAgICAgICAgIG5hbWUgPSBvcy5wYXRo"
    "LmJhc2VuYW1lKGVbImZpbGUiXSkucmVwbGFjZSgifCIsICLvvI8iKQogICAgICAgICAgICAgICAgTC5hcHBlbmQoZiJ8IHtuYW1l"
    "fSB8IHtlWydlcnJvciddLnJlcGxhY2UoJ3wnLCAn77yPJyl9IHwiKQogICAgICAgICAgICBMICs9IFsiIiwgIi0tLSIsICIiLAog"
    "ICAgICAgICAgICAgICAgICAiIyMjIOyekOyjvCDrgpjsmKTripQg6rK97JqwIiwgIiIsCiAgICAgICAgICAgICAgICAgICIqKuyY"
    "myDsmKTtlLzsiqQg7ZiV7IudKGAueGxzYCBgLnBwdGAgYC5kb2NgKSoqIiwKICAgICAgICAgICAgICAgICAgIu2VtOuLuSDtlITr"
    "oZzqt7jrnqjsl5DshJwg7Je07Ja0ICoq64uk66W4IOydtOumhOycvOuhnCDsoIDsnqUqKiDihpIgIgogICAgICAgICAgICAgICAg"
    "ICAiYC54bHN4YCBgLnBwdHhgIGAuZG9jeGAg66GcIOuwlOq+vCDrkqQg64uk7IucIOuzgO2ZmO2VmOyEuOyalC4iLAogICAgICAg"
    "ICAgICAgICAgICAi7ZmV7J6l7J6Q66eMIOuwlOq/lCDsk7Qg7YyM7J2864+EIOqwmeydgCDsmKTrpZjqsIAg64Kp64uI64ukLiIs"
    "CiAgICAgICAgICAgICAgICAgICIiLAogICAgICAgICAgICAgICAgICAiKirquIDsnpDqsIAg7JeG64qUIFBERioqIiwKICAgICAg"
    "ICAgICAgICAgICAgIuyiheydtOulvCDsiqTsupTtlZjqsbDrgpgg7IKs7KeE7Jy866GcIOunjOuToCDrrLjshJzsnoXri4jri6Qu"
    "ICIKICAgICAgICAgICAgICAgICAgIuydtCDrj4TqtazripQgKirquIDsnpAg7J247IudKE9DUinsnYQg7ZWY7KeAIOyViuycvOuv"
    "gOuhnCoqIOuzgO2ZmO2VoCDsiJgg7JeG7Iq164uI64ukLiIsCiAgICAgICAgICAgICAgICAgICLsm5Drs7gg66y47IScIO2MjOyd"
    "vOydtCDsnojsnLzrqbQg6re46rKD7J2EIOuzgO2ZmO2VmOyEuOyalC4iXQogICAgICAgICAgICB3aXRoIGlvLm9wZW4ob3MucGF0"
    "aC5qb2luKHNlbGYub3V0LCAiX+yYpOulmC5tZCIpLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgICAgICAg"
    "ICBmLndyaXRlKCJcbiIuam9pbihMKSkK"
  ),
  "pkos_gdrive.py": (
    "IyAtKi0gY29kaW5nOiB1dGYtOCAtKi0KIiIiClBLT1Mg6rWs6riAIOusuOyEnCDqsIDsoLjsmKTquLAKPT09PT09PT09PT09PT09"
    "PT09PT09PT09PQrqtazquIAg66y47IScwrfsi5ztirjCt+yKrOudvOydtOuTnOuKlCAn64K0IOy7tO2TqO2EsOyXkCDsi6TssrTq"
    "sIAg7JeG64qUJyDsmKjrnbzsnbgg66y47ISc65287IScCu2MjOydvOuhnOuKlCDsnb3snYQg7IiYIOyXhuuLpC4gRHJpdmUgQVBJ"
    "IOuhnCDrgrTrs7TrgrTquLAoZXhwb3J0KSDtlbTslbwg7ZWc64ukLgoK7L2U656p7JeQ7IScIOyTsOuKlCDqsoPsnYQg7KCE7KCc"
    "66GcIO2VnOuLpCAo67OE64+EIOyduOymnSDshKTsoJUg7JeG7J20IOuzuOyduCDqs4TsoJXsnLzroZwg64+Z7J6RKS4KCiAgICBm"
    "cm9tIHBrb3NfZ2RyaXZlIGltcG9ydCBHb29nbGVEb2NzCgogICAgZyA9IEdvb2dsZURvY3MoKSAgICAgICAgICAgICAgICAgICAg"
    "ICMg7J247KadCiAgICBnLmxpc3RfZm9sZGVyKCIxQWJDLi4uIikgICAgICAgICAgICAgIyDtj7TrjZQg7JWIIOq1rOq4gCDrrLjs"
    "hJwg66qp66GdCiAgICBnLmV4cG9ydF9mb2xkZXIoIjFBYkMuLi4iLCAiL2NvbnRlbnQvZHJpdmUvTXlEcml2ZS9QS09TL+q1rOq4"
    "gOusuOyEnCIpCgpQS09TKOqwnOyduOyngOyLneyatOyYgeyytOqzhCkg7ZSE66Gc7KCd7Yq4CiIiIgoKZnJvbSBfX2Z1dHVyZV9f"
    "IGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBpbwppbXBvcnQgcmUKaW1wb3J0IGpzb24KaW1wb3J0IHRpbWUK"
    "CiMg6rWs6riAIOusuOyEnCDsooXrpZggLT4g7Ja065akIO2YleyLneycvOuhnCDrsJvslYTsmKzsp4AKIwojICAg66y47IScICAg"
    "OiDqtazquIDsnbQg66eI7YGs64uk7Jq07Jy866GcIOuwlOuhnCDrgrTrs7TrgrQg7KSA64ukCiMgICDsi5ztirggICA6IGNzdiDr"
    "oZwg67Cb7Jy866m0ICfssqsg7J6lJ+unjCDsmKjri6QgLT4geGxzeCDroZwg67Cb7JWEIOuqqOuToCDsi5ztirjrpbwg7ZGc66Gc"
    "IOyYruq4tOuLpAojICAg7Iqs65287J2065OcOiB0eHQg66GcIOuwm+ycvOuptCDrsJztkZzsnpAg64W47Yq46rCAIOu5oOynhOuL"
    "pCAtPiBwcHR4IOuhnCDrsJvslYQg64W47Yq46rmM7KeAIOyYruq4tOuLpAojCiMg7Ja065akIOqyveyasOuToCDstZzsooUg6rKw"
    "6rO864qUIC5tZCDtlZjrgpjroZwg7Ya17J287ZWc64ukLgpYTFNYX01JTUUgPSAiYXBwbGljYXRpb24vdm5kLm9wZW54bWxmb3Jt"
    "YXRzLW9mZmljZWRvY3VtZW50LnNwcmVhZHNoZWV0bWwuc2hlZXQiClBQVFhfTUlNRSA9ICJhcHBsaWNhdGlvbi92bmQub3Blbnht"
    "bGZvcm1hdHMtb2ZmaWNlZG9jdW1lbnQucHJlc2VudGF0aW9ubWwucHJlc2VudGF0aW9uIgoKRVhQT1JUX0FTID0gewogICAgImFw"
    "cGxpY2F0aW9uL3ZuZC5nb29nbGUtYXBwcy5kb2N1bWVudCI6ICAgICAoInRleHQvbWFya2Rvd24iLCAiLm1kIiwgIuq1rOq4gOus"
    "uOyEnCIpLAogICAgImFwcGxpY2F0aW9uL3ZuZC5nb29nbGUtYXBwcy5zcHJlYWRzaGVldCI6ICAoWExTWF9NSU1FLCAiLm1kIiwg"
    "Iuq1rOq4gOyLnO2KuCIpLAogICAgImFwcGxpY2F0aW9uL3ZuZC5nb29nbGUtYXBwcy5wcmVzZW50YXRpb24iOiAoUFBUWF9NSU1F"
    "LCAiLm1kIiwgIuq1rOq4gOyKrOudvOydtOuTnCIpLAp9CgpGT0xERVJfTUlNRSA9ICJhcHBsaWNhdGlvbi92bmQuZ29vZ2xlLWFw"
    "cHMuZm9sZGVyIgoKX0lOVkFMSUQgPSByZS5jb21waWxlKHInW1xcLzoqPyI8PnxdJykKCgpkZWYgc2FmZV9uYW1lKG5hbWU6IHN0"
    "ciwgbWF4bGVuOiBpbnQgPSA5MCkgLT4gc3RyOgogICAgcmV0dXJuIChfSU5WQUxJRC5zdWIoIl8iLCBuYW1lKS5zdHJpcCgpWzpt"
    "YXhsZW5dLnJzdHJpcCgiIC4iKSkgb3IgIuustOygnCIKCgpkZWYgZm9sZGVyX2lkX2Zyb20odGV4dDogc3RyKSAtPiBzdHIgfCBO"
    "b25lOgogICAgIiIi7Y+0642UIOunge2BrCDrmJDripQgSUQg66y47J6Q7Je07JeQ7IScIElE66eMIOu9keyVhOuCuOuLpC4iIiIK"
    "ICAgIHRleHQgPSAodGV4dCBvciAiIikuc3RyaXAoKQogICAgbSA9IHJlLnNlYXJjaChyIi9mb2xkZXJzLyhbQS1aYS16MC05Xy1d"
    "ezEwLH0pIiwgdGV4dCkKICAgIGlmIG06CiAgICAgICAgcmV0dXJuIG0uZ3JvdXAoMSkKICAgIG0gPSByZS5tYXRjaChyIl4oW0Et"
    "WmEtejAtOV8tXXsxMCx9KSQiLCB0ZXh0KQogICAgcmV0dXJuIG0uZ3JvdXAoMSkgaWYgbSBlbHNlIE5vbmUKCgpjbGFzcyBHb29n"
    "bGVEb2NzOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHZlcmJvc2U6IGJvb2wgPSBUcnVlKToKICAgICAgICBzZWxmLnZlcmJvc2Ug"
    "PSB2ZXJib3NlCiAgICAgICAgc2VsZi5zdmMgPSBzZWxmLl9jb25uZWN0KCkKCiAgICBkZWYgbG9nKHNlbGYsICphKToKICAgICAg"
    "ICBpZiBzZWxmLnZlcmJvc2U6CiAgICAgICAgICAgIHByaW50KCphLCBmbHVzaD1UcnVlKQoKICAgIGRlZiBfY29ubmVjdChzZWxm"
    "KToKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZyb20gZ29vZ2xlLmNvbGFiIGltcG9ydCBhdXRoCiAgICAgICAgICAgIGF1dGgu"
    "YXV0aGVudGljYXRlX3VzZXIoKQogICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICAgICAgcGFzcyAgICAgICAgICAg"
    "ICAgICAgICAgICAgIyDsvZTrnqnsnbQg7JWE64uI66m0IOq4sOuzuCDsnpDqsqnspp3rqoXsnYQg7IKs7JqpCiAgICAgICAgZnJv"
    "bSBnb29nbGVhcGljbGllbnQuZGlzY292ZXJ5IGltcG9ydCBidWlsZAogICAgICAgIHJldHVybiBidWlsZCgiZHJpdmUiLCAidjMi"
    "KQoKICAgICMg4pSA4pSAIO2PtOuNlCDslYgg7ZWt66qpIOuCmOyXtCAo7ZWY7JyEIO2PtOuNlOq5jOyngCkKICAgIGRlZiB3YWxr"
    "KHNlbGYsIGZvbGRlcl9pZDogc3RyLCByZWN1cnNpdmU6IGJvb2wgPSBUcnVlLAogICAgICAgICAgICAgX3ByZWZpeDogc3RyID0g"
    "IiIpIC0+IGxpc3RbZGljdF06CiAgICAgICAgaXRlbXMsIHRva2VuID0gW10sIE5vbmUKICAgICAgICB3aGlsZSBUcnVlOgogICAg"
    "ICAgICAgICByZXNwID0gc2VsZi5zdmMuZmlsZXMoKS5saXN0KAogICAgICAgICAgICAgICAgcT1mIid7Zm9sZGVyX2lkfScgaW4g"
    "cGFyZW50cyBhbmQgdHJhc2hlZCA9IGZhbHNlIiwKICAgICAgICAgICAgICAgIGZpZWxkcz0ibmV4dFBhZ2VUb2tlbiwgZmlsZXMo"
    "aWQsbmFtZSxtaW1lVHlwZSxtb2RpZmllZFRpbWUpIiwKICAgICAgICAgICAgICAgIHBhZ2VTaXplPTIwMCwgcGFnZVRva2VuPXRv"
    "a2VuLAogICAgICAgICAgICAgICAgc3VwcG9ydHNBbGxEcml2ZXM9VHJ1ZSwgaW5jbHVkZUl0ZW1zRnJvbUFsbERyaXZlcz1UcnVl"
    "LAogICAgICAgICAgICApLmV4ZWN1dGUoKQogICAgICAgICAgICBmb3IgZiBpbiByZXNwLmdldCgiZmlsZXMiLCBbXSk6CiAgICAg"
    "ICAgICAgICAgICBmWyJwYXRoIl0gPSBvcy5wYXRoLmpvaW4oX3ByZWZpeCwgc2FmZV9uYW1lKGZbIm5hbWUiXSkpCiAgICAgICAg"
    "ICAgICAgICBpZiBmWyJtaW1lVHlwZSJdID09IEZPTERFUl9NSU1FOgogICAgICAgICAgICAgICAgICAgIGlmIHJlY3Vyc2l2ZToK"
    "ICAgICAgICAgICAgICAgICAgICAgICAgaXRlbXMgKz0gc2VsZi53YWxrKGZbImlkIl0sIFRydWUsIGZbInBhdGgiXSkKICAgICAg"
    "ICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgaXRlbXMuYXBwZW5kKGYpCiAgICAgICAgICAgIHRva2VuID0gcmVz"
    "cC5nZXQoIm5leHRQYWdlVG9rZW4iKQogICAgICAgICAgICBpZiBub3QgdG9rZW46CiAgICAgICAgICAgICAgICBicmVhawogICAg"
    "ICAgIHJldHVybiBpdGVtcwoKICAgICMg4pSA4pSAIOuqqeuhnSDrs7TquLAKICAgIGRlZiBsaXN0X2ZvbGRlcihzZWxmLCBmb2xk"
    "ZXJfb3JfbGluazogc3RyLCByZWN1cnNpdmU6IGJvb2wgPSBUcnVlKSAtPiBsaXN0W2RpY3RdOgogICAgICAgIGZpZCA9IGZvbGRl"
    "cl9pZF9mcm9tKGZvbGRlcl9vcl9saW5rKQogICAgICAgIGlmIG5vdCBmaWQ6CiAgICAgICAgICAgIHNlbGYubG9nKCLtj7TrjZQg"
    "66eB7YGsIOuYkOuKlCBJRCDtmJXsi53snbQg7JWE64uZ64uI64ukLiIpCiAgICAgICAgICAgIHJldHVybiBbXQogICAgICAgIGZp"
    "bGVzID0gc2VsZi53YWxrKGZpZCwgcmVjdXJzaXZlKQogICAgICAgIGdvb2dsZSA9IFtmIGZvciBmIGluIGZpbGVzIGlmIGZbIm1p"
    "bWVUeXBlIl0gaW4gRVhQT1JUX0FTXQogICAgICAgIG90aGVyID0gW2YgZm9yIGYgaW4gZmlsZXMgaWYgZlsibWltZVR5cGUiXSBu"
    "b3QgaW4gRVhQT1JUX0FTXQoKICAgICAgICBzZWxmLmxvZyhmIuyghOyytCB7bGVuKGZpbGVzKX3qsJwiKQogICAgICAgIHNlbGYu"
    "bG9nKGYiICDqsIDsoLjsmKwg7IiYIOyeiOuKlCDqtazquIAg66y47IScIDoge2xlbihnb29nbGUpfeqwnCIpCiAgICAgICAgY291"
    "bnRzID0ge30KICAgICAgICBmb3IgZiBpbiBnb29nbGU6CiAgICAgICAgICAgIGsgPSBFWFBPUlRfQVNbZlsibWltZVR5cGUiXV1b"
    "Ml0KICAgICAgICAgICAgY291bnRzW2tdID0gY291bnRzLmdldChrLCAwKSArIDEKICAgICAgICBmb3IgaywgbiBpbiBjb3VudHMu"
    "aXRlbXMoKToKICAgICAgICAgICAgc2VsZi5sb2coZiIgICAgIHtrOjEyfSB7bn3qsJwiKQogICAgICAgIHNlbGYubG9nKGYiICDs"
    "nbzrsJgg7YyM7J28KOuzhOuPhCDrs4DtmZgg7ZWE7JqUKSA6IHtsZW4ob3RoZXIpfeqwnCIpCiAgICAgICAgcmV0dXJuIGdvb2ds"
    "ZQoKICAgICMg4pSA4pSAIO2VnCDqsJwg64K067O064K06riwICjrrLTsl4fsnbTrk6AgLm1kIOuhnCDrp4zrk6Dri6QpCiAgICBk"
    "ZWYgZXhwb3J0X29uZShzZWxmLCBmaWxlX2lkOiBzdHIsIG1pbWU6IHN0ciwgZHN0X25vZXh0OiBzdHIpIC0+IHN0ciB8IE5vbmU6"
    "CiAgICAgICAgdGFyZ2V0LCBleHQsIF9raW5kID0gRVhQT1JUX0FTW21pbWVdCgogICAgICAgIHRyeToKICAgICAgICAgICAgZGF0"
    "YSA9IHNlbGYuc3ZjLmZpbGVzKCkuZXhwb3J0KGZpbGVJZD1maWxlX2lkLCBtaW1lVHlwZT10YXJnZXQpLmV4ZWN1dGUoKQogICAg"
    "ICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICMg7JqU7LKt7ZWcIO2YleyLneydhCDsp4Dsm5DtlZjsp4Ag7JWK7Jy8"
    "66m0IOydvOuwmCDthY3siqTtirjroZwg7ZuE7Ye0CiAgICAgICAgICAgIGRhdGEgPSBzZWxmLnN2Yy5maWxlcygpLmV4cG9ydChm"
    "aWxlSWQ9ZmlsZV9pZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1pbWVUeXBlPSJ0ZXh0L3Bs"
    "YWluIikuZXhlY3V0ZSgpCiAgICAgICAgICAgIHRhcmdldCA9ICJ0ZXh0L3BsYWluIgoKICAgICAgICBpZiBub3QgaXNpbnN0YW5j"
    "ZShkYXRhLCBieXRlcyk6CiAgICAgICAgICAgIGRhdGEgPSBkYXRhLmVuY29kZSgidXRmLTgiKQoKICAgICAgICBwYXRoID0gZHN0"
    "X25vZXh0ICsgZXh0CiAgICAgICAgb3MubWFrZWRpcnMob3MucGF0aC5kaXJuYW1lKHBhdGgpLCBleGlzdF9vaz1UcnVlKQoKICAg"
    "ICAgICBpZiB0YXJnZXQgaW4gKFhMU1hfTUlNRSwgUFBUWF9NSU1FKToKICAgICAgICAgICAgYm9keSA9IHNlbGYuX29mZmljZV90"
    "b19tZChkYXRhLCB0YXJnZXQpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYm9keSA9IGRhdGEuZGVjb2RlKCJ1dGYtOCIsICJp"
    "Z25vcmUiKQoKICAgICAgICB3aXRoIGlvLm9wZW4ocGF0aCwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAg"
    "ICBmLndyaXRlKGJvZHkpCiAgICAgICAgcmV0dXJuIHBhdGgKCiAgICBAc3RhdGljbWV0aG9kCiAgICBkZWYgX29mZmljZV90b19t"
    "ZChkYXRhOiBieXRlcywgdGFyZ2V0OiBzdHIpIC0+IHN0cjoKICAgICAgICAiIiJ4bHN4L3BwdHgg7JuQ67O47J2EIOydtOuvuCDq"
    "soDspp3rkJwg7J296riwIOuqqOuTiOuhnCDrp4jtgazri6TsmrTsnLzroZwg7Jiu6ri064ukLiIiIgogICAgICAgIGltcG9ydCB0"
    "ZW1wZmlsZQogICAgICAgIGZyb20gcGtvc19yZWFkZXJzIGltcG9ydCByZWFkX3hsc3gsIHJlYWRfcHB0eAoKICAgICAgICBzdWZm"
    "aXggPSAiLnhsc3giIGlmIHRhcmdldCA9PSBYTFNYX01JTUUgZWxzZSAiLnBwdHgiCiAgICAgICAgdG1wID0gdGVtcGZpbGUuTmFt"
    "ZWRUZW1wb3JhcnlGaWxlKHN1ZmZpeD1zdWZmaXgsIGRlbGV0ZT1GYWxzZSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIHRtcC53"
    "cml0ZShkYXRhKQogICAgICAgICAgICB0bXAuY2xvc2UoKQogICAgICAgICAgICByZXMgPSByZWFkX3hsc3godG1wLm5hbWUpIGlm"
    "IHN1ZmZpeCA9PSAiLnhsc3giIGVsc2UgcmVhZF9wcHR4KHRtcC5uYW1lKQogICAgICAgICAgICByZXR1cm4gcmVzLnRleHQgaWYg"
    "cmVzLm9rIGVsc2UgZiI+IOuCtOyaqeydhCDsnb3sp4Ag66q77ZaI7Iq164uI64ukOiB7cmVzLmVycm9yfSIKICAgICAgICBmaW5h"
    "bGx5OgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBvcy51bmxpbmsodG1wLm5hbWUpCiAgICAgICAgICAgIGV4Y2Vw"
    "dCBPU0Vycm9yOgogICAgICAgICAgICAgICAgcGFzcwoKICAgICMg4pSA4pSAIO2PtOuNlCDthrXsp7jroZwg64K067O064K06riw"
    "CiAgICBkZWYgZXhwb3J0X2ZvbGRlcihzZWxmLCBmb2xkZXJfb3JfbGluazogc3RyLCBvdXRfZGlyOiBzdHIsCiAgICAgICAgICAg"
    "ICAgICAgICAgICByZWN1cnNpdmU6IGJvb2wgPSBUcnVlLCBza2lwX2V4aXN0aW5nOiBib29sID0gVHJ1ZSkgLT4gZGljdDoKICAg"
    "ICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgZmlsZXMgPSBzZWxmLmxpc3RfZm9sZGVyKGZvbGRlcl9vcl9saW5rLCByZWN1"
    "cnNpdmUpCiAgICAgICAgaWYgbm90IGZpbGVzOgogICAgICAgICAgICAjIO2CpOulvCDruaDrnKjrpqzrqbQg6rKw6rO866W8IOuw"
    "m+yVhCDsk7DripQg7Kq97JeQ7IScIOyYpOulmOqwgCDrgpzri6QKICAgICAgICAgICAgcmV0dXJuIHsiZG9uZSI6IDAsICJza2lw"
    "cGVkIjogMCwgImZhaWxlZCI6IDB9CgogICAgICAgIHNlbGYubG9nKGYiXG57b3V0X2Rpcn0g66GcIOqwgOyguOyYteuLiOuLpC5c"
    "biIpCiAgICAgICAgb3MubWFrZWRpcnMob3V0X2RpciwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBkb25lID0gc2tpcHBlZCA9IGZh"
    "aWxlZCA9IDAKICAgICAgICByZWNvcmRzLCBlcnJvcnMgPSBbXSwgW10KCiAgICAgICAgZm9yIGksIGYgaW4gZW51bWVyYXRlKGZp"
    "bGVzLCAxKToKICAgICAgICAgICAga2luZCA9IEVYUE9SVF9BU1tmWyJtaW1lVHlwZSJdXVsyXQogICAgICAgICAgICBkc3Rfbm9l"
    "eHQgPSBvcy5wYXRoLmpvaW4ob3V0X2RpciwgZlsicGF0aCJdKQogICAgICAgICAgICBndWVzcyA9IGRzdF9ub2V4dCArIEVYUE9S"
    "VF9BU1tmWyJtaW1lVHlwZSJdXVsxXQogICAgICAgICAgICBpZiBza2lwX2V4aXN0aW5nIGFuZCBvcy5wYXRoLmV4aXN0cyhndWVz"
    "cyk6CiAgICAgICAgICAgICAgICBza2lwcGVkICs9IDEKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHRyeToK"
    "ICAgICAgICAgICAgICAgIHBhdGggPSBzZWxmLmV4cG9ydF9vbmUoZlsiaWQiXSwgZlsibWltZVR5cGUiXSwgZHN0X25vZXh0KQog"
    "ICAgICAgICAgICAgICAgc2VsZi5fYWRkX2Zyb250X21hdHRlcihwYXRoLCBmLCBraW5kKQogICAgICAgICAgICAgICAgcmVjb3Jk"
    "cy5hcHBlbmQoeyJ0aXRsZSI6IGZbIm5hbWUiXSwgImtpbmQiOiBraW5kLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAg"
    "ICJmaWxlIjogb3MucGF0aC5yZWxwYXRoKHBhdGgsIG91dF9kaXIpLnJlcGxhY2UoIlxcIiwgIi8iKSwKICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAgICAgICAiZGF0ZSI6IGYuZ2V0KCJtb2RpZmllZFRpbWUiLCAiIilbOjEwXSwKICAgICAgICAgICAgICAgICAg"
    "ICAgICAgICAgICAgICAidXJsIjogZiJodHRwczovL2RyaXZlLmdvb2dsZS5jb20vb3Blbj9pZD17ZlsnaWQnXX0ifSkKICAgICAg"
    "ICAgICAgICAgIGRvbmUgKz0gMQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICBlcnJv"
    "cnMuYXBwZW5kKHsibmFtZSI6IGZbIm5hbWUiXSwgImVycm9yIjogc3RyKGUpWzoyMDBdfSkKICAgICAgICAgICAgICAgIGZhaWxl"
    "ZCArPSAxCiAgICAgICAgICAgIGlmIGRvbmUgYW5kIGRvbmUgJSAyMCA9PSAwOgogICAgICAgICAgICAgICAgc2VsZi5sb2coZiIg"
    "IOKApiB7ZG9uZX3qsJwgKHtpfS97bGVuKGZpbGVzKX0pIikKCiAgICAgICAgc2VsZi5fd3JpdGVfaW5kZXgob3V0X2RpciwgcmVj"
    "b3JkcywgZXJyb3JzKQogICAgICAgIHNlY3MgPSBpbnQodGltZS50aW1lKCkgLSB0MCkKICAgICAgICBzZWxmLmxvZyhmIlxu7JmE"
    "66OMISDqsIDsoLjsmLQge2RvbmV96rCcIMK3IOqxtOuEiOucgCB7c2tpcHBlZH3qsJwgwrcg7Iuk7YyoIHtmYWlsZWR96rCcICIK"
    "ICAgICAgICAgICAgICAgICBmIsK3IHtzZWNzIC8vIDYwfeu2hCB7c2VjcyAlIDYwfey0iCIpCiAgICAgICAgcmV0dXJuIHsiZG9u"
    "ZSI6IGRvbmUsICJza2lwcGVkIjogc2tpcHBlZCwgImZhaWxlZCI6IGZhaWxlZH0KCiAgICAjIOKUgOKUgCDrgrTrs7Trgrgg7YyM"
    "7J28IOyVnuyXkCDsoJXrs7Qg67aZ7J206riwCiAgICBkZWYgX2FkZF9mcm9udF9tYXR0ZXIoc2VsZiwgcGF0aDogc3RyLCBmOiBk"
    "aWN0LCBraW5kOiBzdHIpOgogICAgICAgIHRyeToKICAgICAgICAgICAgd2l0aCBpby5vcGVuKHBhdGgsIGVuY29kaW5nPSJ1dGYt"
    "OCIpIGFzIGZoOgogICAgICAgICAgICAgICAgYm9keSA9IGZoLnJlYWQoKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAg"
    "ICAgICAgIHJldHVybgogICAgICAgIHRpdGxlID0gZlsibmFtZSJdLnJlcGxhY2UoJyInLCAiJyIpCiAgICAgICAgaGVhZCA9ICgi"
    "LS0tXG4iCiAgICAgICAgICAgICAgICBmJ3RpdGxlOiAie3RpdGxlfSJcbicKICAgICAgICAgICAgICAgIGYnZGF0ZToge2YuZ2V0"
    "KCJtb2RpZmllZFRpbWUiLCIiKVs6MTBdfVxuJwogICAgICAgICAgICAgICAgZidraW5kOiAie2tpbmR9IlxuJwogICAgICAgICAg"
    "ICAgICAgZid1cmw6IGh0dHBzOi8vZHJpdmUuZ29vZ2xlLmNvbS9vcGVuP2lkPXtmWyJpZCJdfVxuJwogICAgICAgICAgICAgICAg"
    "Ii0tLVxuXG4iCiAgICAgICAgICAgICAgICBmIiMge3RpdGxlfVxuXG4iKQogICAgICAgIHdpdGggaW8ub3BlbihwYXRoLCAidyIs"
    "IGVuY29kaW5nPSJ1dGYtOCIpIGFzIGZoOgogICAgICAgICAgICBmaC53cml0ZShoZWFkICsgYm9keSkKCiAgICBkZWYgX3dyaXRl"
    "X2luZGV4KHNlbGYsIG91dF9kaXI6IHN0ciwgcmVjb3JkczogbGlzdCwgZXJyb3JzOiBsaXN0KToKICAgICAgICBpZiByZWNvcmRz"
    "OgogICAgICAgICAgICB3aXRoIGlvLm9wZW4ob3MucGF0aC5qb2luKG91dF9kaXIsICJfZmlsZXMuanNvbiIpLCAidyIsIGVuY29k"
    "aW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgICAgICAgICBqc29uLmR1bXAocmVjb3JkcywgZiwgZW5zdXJlX2FzY2lpPUZhbHNl"
    "LCBpbmRlbnQ9MSkKICAgICAgICAgICAgTCA9IFsiIyDwn5OEIOqwgOyguOyYqCDqtazquIAg66y47IScIiwgIiIsIGYi7KCE7LK0"
    "ICoqe2xlbihyZWNvcmRzKX3qsJwqKiIsICIiXQogICAgICAgICAgICBmb3IgciBpbiBzb3J0ZWQocmVjb3Jkcywga2V5PWxhbWJk"
    "YSB4OiB4WyJmaWxlIl0pOgogICAgICAgICAgICAgICAgTC5hcHBlbmQoZiItIFt7clsndGl0bGUnXX1dKHtyWydmaWxlJ119KSDC"
    "tyB7clsna2luZCddfSDCtyB7clsnZGF0ZSddfSIpCiAgICAgICAgICAgIHdpdGggaW8ub3Blbihvcy5wYXRoLmpvaW4ob3V0X2Rp"
    "ciwgIklOREVYLm1kIiksICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgICAgIGYud3JpdGUoIlxuIi5q"
    "b2luKEwpKQogICAgICAgIGlmIGVycm9yczoKICAgICAgICAgICAgTCA9IFsiIyDimqAg6rCA7KC47Jik7KeAIOuqu+2VnCDrrLjs"
    "hJwiLCAiIiwgInwg66y47IScIHwg7J207JygIHwiLCAifC0tLS0tLXwtLS0tLS18Il0KICAgICAgICAgICAgZm9yIGUgaW4gZXJy"
    "b3JzOgogICAgICAgICAgICAgICAgTC5hcHBlbmQoZiJ8IHtlWyduYW1lJ10ucmVwbGFjZSgnfCcsJ++8jycpfSB8IHtlWydlcnJv"
    "ciddfSB8IikKICAgICAgICAgICAgd2l0aCBpby5vcGVuKG9zLnBhdGguam9pbihvdXRfZGlyLCAiX+yYpOulmC5tZCIpLCAidyIs"
    "IGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgICAgICAgICBmLndyaXRlKCJcbiIuam9pbihMKSkK"
  ),
}

for _name, _b64 in _ENGINES.items():
    pathlib.Path(_name).write_bytes(base64.b64decode(_b64))
sys.path.insert(0, ".")

import pkos_converter, pkos_readers, pkos_privacy, pkos_folder, pkos_paths
for _m in (pkos_converter, pkos_readers, pkos_privacy, pkos_folder, pkos_paths):
    importlib.reload(_m)
from pkos_converter import Converter, Settings, inspect
from pkos_readers import read_any, SUPPORTED
from pkos_privacy import PrivacyFilter, Policy, preview as 개인정보_미리보기
from pkos_folder import FolderConverter, FolderSettings
from pkos_paths import drive_path

print("엔진 준비 완료!")
print("다룰 수 있는 형식:", " ".join(SUPPORTED))

## 먼저 연습하기 · 태극기와 애국가 샘플

준비하기 두 칸을 실행한 다음, 아래 **▶ 샘플 PDF로 변환 테스트**를 누르세요.
폴더 경로 입력 없이 2쪽짜리 공개 샘플을 만들고, 2부와 같은 변환기로 마크다운을 생성합니다.

- 원본: 태극기 이미지 + 애국가 1~4절과 후렴. 음원이나 악보는 포함하지 않습니다.
- 결과: 제목과 가사 등 **글자만** 변환합니다. 현재 2부 PDF 변환기는 그림을 추출하지 않습니다.
- 실행 후 변환 결과가 아래에 표시됩니다. 태극기는 왼쪽 📁에서 `PKOS_샘플/원본`의 PDF를 내려받아 확인하세요.
- 원본과 결과는 코랩 임시 공간 `/content/PKOS_샘플`에 저장됩니다. 런타임이 삭제되면 사라지므로 보관하려면 파일을 다운로드하세요.
- 이 공개 샘플은 원문 비교를 위해 개인정보 가리기를 끕니다. 개인 자료를 변환할 때는 2부의 개인정보 설정을 사용하세요.


In [ ]:
#@title ▶ 샘플 PDF로 변환 테스트
import base64
from pathlib import Path
from IPython.display import display, Markdown
from pkos_folder import FolderConverter, FolderSettings

sample_root = Path("/content/PKOS_샘플")
sample_src = sample_root / "원본"
sample_out = sample_root / "변환결과"
sample_src.mkdir(parents=True, exist_ok=True)
sample_pdf = sample_src / "대한민국_태극기와_애국가.pdf"
sample_pdf.write_bytes(base64.b64decode("..."))
sample_fc = FolderConverter(FolderSettings(
    src_dir=str(sample_src), out_dir=str(sample_out),
    skip_existing=False, 개인정보_가리기=False,
))
sample_result = sample_fc.run()
print("샘플 PDF:", sample_pdf)
print("변환 결과:", sample_out)
for record in sample_fc.records:
    result_path = sample_out / record["md"]
    display(Markdown(result_path.read_text(encoding="utf-8")))


---
---

# 📝 1부 · 네이버 블로그 백업 PDF 변환

블로그 글이 **한 편씩 따로** 마크다운 파일이 됩니다. (제목·날짜·카테고리·원문주소 포함)

### 미리 준비할 것

1. **블로그를 PDF로 백업**
   블로그 관리 → 글 전체보기 → 인쇄 → 대상을 **'PDF로 저장'**
   (100편 정도씩 나눠 저장하면 안정적입니다)
2. 구글 드라이브에 폴더를 만들고 PDF 넣기

*(블로그가 없으면 이 부는 건너뛰고 2부로 가세요)*

## 1-① PDF가 들어있는 폴더 알려주기

아래 **세 가지 방법 중 아무거나** 하나만 채우면 됩니다.

| 방법 | 예시 |
|------|------|
| 폴더 **링크** 붙여넣기 | `https://drive.google.com/drive/folders/1AbC...` |
| 폴더 **ID**만 붙여넣기 | `1AbCdEfGhIjK...` |
| 내 드라이브 안 **경로** | `블로그백업` 또는 `기록/블로그백업` |

In [ ]:
#@title ▶ 폴더 지정하기 { display-mode: "form" }
#@markdown ### 폴더 링크 또는 ID (둘 중 하나, 없으면 비워두세요)
드라이브_링크_또는_ID = ""  #@param {type:"string"}
#@markdown ### 또는, 내 드라이브 안의 폴더 경로
내_드라이브_경로 = "블로그백업"  #@param {type:"string"}

import os, re

MYDRIVE = "/content/drive/MyDrive"


def _folder_path_from_id(folder_id):
    """드라이브 폴더 ID -> 마운트된 실제 경로"""
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    svc = build("drive", "v3")
    parts = []
    cur = folder_id
    for _ in range(20):
        info = svc.files().get(fileId=cur, fields="id,name,parents").execute()
        parts.append(info["name"])
        parents = info.get("parents")
        if not parents:
            break
        cur = parents[0]
    parts.reverse()
    # 최상위(내 드라이브) 이름은 버리고 이어붙인다
    return os.path.join(MYDRIVE, *parts[1:]) if len(parts) > 1 else MYDRIVE


PDF_DIR = None
raw = 드라이브_링크_또는_ID.strip()

if raw:
    m = re.search(r"/folders/([A-Za-z0-9_-]{10,})", raw) or re.match(r"^([A-Za-z0-9_-]{10,})$", raw)
    if not m:
        print("링크/ID 형식을 알아보지 못했습니다. 폴더 주소를 그대로 붙여넣어 보세요.")
    else:
        try:
            PDF_DIR = _folder_path_from_id(m.group(1))
            print(f"폴더를 찾았습니다: {PDF_DIR}")
        except Exception as e:
            print(f"ID로 찾기 실패({e}). 아래 '경로' 방식을 써주세요.")

if PDF_DIR is None:
    PDF_DIR = drive_path(내_드라이브_경로)

print()
if os.path.isdir(PDF_DIR):
    pdfs = sorted(n for n in os.listdir(PDF_DIR) if n.lower().endswith(".pdf"))
    print(f"경로 : {PDF_DIR}")
    print(f"PDF  : {len(pdfs)}개 발견")
    for n in pdfs[:15]:
        mb = os.path.getsize(os.path.join(PDF_DIR, n)) / 1024 / 1024
        print(f"   - {n}  ({mb:.1f} MB)")
    if len(pdfs) > 15:
        print(f"   … 외 {len(pdfs)-15}개")
    if not pdfs:
        print("\n⚠ 이 폴더에 PDF가 없습니다. 폴더를 다시 확인해주세요.")
else:
    print(f"⚠ 폴더를 찾을 수 없습니다: {PDF_DIR}")
    print("   왼쪽 파일 탐색기(📁)에서 실제 폴더 이름을 확인해보세요.")

## 1-② 미리 확인하기 (권장)

변환하기 전에, PDF가 올바른 형식인지 **미리 훑어봅니다.**
글이 몇 편 들어있는지 여기서 확인할 수 있어요.

In [ ]:
#@title ▶ 미리 확인하기 { display-mode: "form" }
import os

pdfs = sorted(n for n in os.listdir(PDF_DIR) if n.lower().endswith(".pdf"))
총합 = 0
for n in pdfs:
    r = inspect(os.path.join(PDF_DIR, n))
    총합 += r["posts"]
    print("-" * 46)
print(f"\n예상 변환 결과: 전체 약 {총합}편")

## 1-③ 변환 실행

설정을 확인하고 ▶ 를 누르세요. PDF 양에 따라 몇 분 걸립니다.

In [ ]:
#@title ▶ 변환 시작 { display-mode: "form" }
#@markdown ### 결과를 저장할 폴더 이름 (PDF 폴더 안에 생깁니다)
저장폴더 = "md"  #@param {type:"string"}
#@markdown ### 본문 사진도 함께 저장할까요?
사진_저장 = True  #@param {type:"boolean"}
#@markdown ### 이미 변환한 글은 건너뛸까요? (다시 돌려도 안전)
중복_건너뛰기 = True  #@param {type:"boolean"}
#@markdown ### 파일 이름 형식
파일이름형식 = "{date}_{title}"  #@param ["{date}_{title}", "{title}", "{date}"]

import os

OUT_DIR = os.path.join(PDF_DIR, 저장폴더.strip() or "md")

conv = Converter(Settings(
    pdf_dir          = PDF_DIR,
    out_dir          = OUT_DIR,
    extract_images   = 사진_저장,
    skip_existing    = 중복_건너뛰기,
    filename_pattern = 파일이름형식,
))
결과 = conv.run()

print()
print("저장 위치:", OUT_DIR)
print("구글 드라이브에 반영되기까지 잠시 걸릴 수 있습니다.")

## 1-④ 결과 살펴보기

In [ ]:
#@title ▶ 결과 요약 보기 { display-mode: "form" }
import os, io, json, collections

idx_path = os.path.join(OUT_DIR, "_index.json")
with io.open(idx_path, encoding="utf-8") as f:
    idx = json.load(f)

print(f"전체 {len(idx)}편\n")

years = collections.Counter(e["date"][:4] for e in idx)
print("연도별")
for y in sorted(years):
    print(f"   {y} : {years[y]:4}편  " + "█" * min(40, years[y] // 3))

print("\n카테고리 상위 10")
for c, n in collections.Counter(e.get("category", "") for e in idx).most_common(10):
    print(f"   {n:4}편  {c or '(없음)'}")

print(f"\n기간 : {min(e['date'] for e in idx)} ~ {max(e['date'] for e in idx)}")
print(f"목차 : {os.path.join(OUT_DIR, 'INDEX.md')}")

---
---

# 📂 2부 · 문서 폴더 통째로 변환

블로그 PDF 말고도, **폴더 하나를 통째로** 마크다운으로 바꿀 수 있습니다.

| 다루는 형식 | |
|---|---|
| 한글 | `.hwp` `.hwpx` |
| 오피스 | `.docx` `.pptx` `.xlsx` |
| 그 외 | `.pdf` `.html` `.txt` `.csv` |

원래 폴더 구조를 그대로 유지하며, 한 파일이 실패해도 나머지는 계속 진행됩니다.

> ⚠️ **개인정보 주의** — 업무 문서에는 이름·연락처·계좌 같은 정보가 들어있을 수 있습니다.
> 변환 결과를 웹에 올릴 때는 **반드시 선별**하세요.

# 코랩에서 변환할 폴더의 경로 복사하기

## 1. 구글 드라이브 연결하기
1. 노트북 맨 위의 **준비하기 → 눌러서 준비하기**에서 ▶를 누릅니다.
2. 연결 승인 창에서 **변환할 자료가 들어 있는 구글 계정**을 선택하고 연결을 허용합니다.
3. 준비 완료 안내가 나오면 **눌러서 엔진 불러오기**도 ▶를 눌러 실행합니다.

## 2. 코랩 파일 탐색기 열기
1. 코랩 화면 **맨 왼쪽 세로 아이콘 중 📁 폴더 아이콘**을 누릅니다. 목차 아이콘과 다릅니다.
2. 파일 목록에서 `drive` 왼쪽의 작은 화살표를 누릅니다.
3. 그 안의 `MyDrive` 왼쪽 화살표를 누릅니다. 여기가 웹 구글 드라이브의 **내 드라이브**입니다.
4. 변환할 폴더가 나올 때까지 상위 폴더를 차례로 펼칩니다.

예를 들어 실제 폴더가 아래 위치에 있다면:

```text
내 드라이브
└── 00_개인지식운영체계(PKOS)
    └── 01_외부 정보
```

`drive → MyDrive → 00_개인지식운영체계(PKOS)`를 펼친 뒤 **01_외부 정보**를 찾습니다.

## 3. 폴더의 경로 복사하기
1. 변환할 **폴더 이름 위에서 마우스 오른쪽 버튼**을 누릅니다.
2. 메뉴의 **경로 복사(Copy path)**를 선택합니다.
3. 복사되는 값은 다음과 같은 형태입니다.

```text
/content/drive/MyDrive/00_개인지식운영체계(PKOS)/01_외부 정보
```

브라우저 주소창의 `https://drive.google.com/...` 주소나 '공유 링크 복사'는 다른 기능입니다. **코랩 왼쪽 파일 탐색기에서 경로를 복사**하세요.

## 4. 문서폴더와 결과폴더 입력하기
1. **2부 → 폴더 훑어보기**로 이동합니다.
2. **문서폴더** 입력칸을 클릭하고 `Ctrl+A`로 기존 내용을 모두 선택합니다.
3. `Ctrl+V`로 복사한 경로를 붙여넣습니다. 수정된 PKOS 버전에서는 전체 경로를 그대로 넣어도 됩니다.
4. **결과폴더**에는 `PKOS/변환결과`를 입력합니다. 변환된 마크다운을 모을 위치이며, 변환할 원본 폴더와 구분하세요. 변환 시 폴더가 생성됩니다.
5. 이 칸 왼쪽의 **▶**를 눌러 목록을 확인합니다. 입력만 바꾸면 아래의 이전 실행 결과는 그대로이므로 반드시 다시 실행하세요.
6. 파일 목록이 확인되면 개인정보 설정과 미리보기를 확인하고 **폴더 변환 시작**을 실행합니다.

수정된 PKOS 버전은 다음 세 가지를 같은 위치로 처리합니다.

| 입력 방식 | 문서폴더에 넣을 값 |
|---|---|
| 코랩에서 경로 복사 | `/content/drive/MyDrive/00_개인지식운영체계(PKOS)/01_외부 정보` |
| 내 드라이브 기준 상대 경로 | `00_개인지식운영체계(PKOS)/01_외부 정보` |
| Windows 탐색기에서 경로 복사 | `G:\내 드라이브\00_개인지식운영체계(PKOS)\01_외부 정보` |

Windows 경로도 코랩에 연결한 계정에 같은 폴더가 있어야 합니다. 이 기능이 PC의 G: 드라이브에 직접 접속하는 것은 아닙니다. 가능하면 **코랩에서 복사한 경로**를 사용하세요.

## 5. 폴더가 보이지 않을 때
- `drive` 자체가 없다면 맨 위 준비하기 셀을 실행해 연결부터 완료하세요.
- `MyDrive`는 있지만 원하는 폴더가 없다면 웹 구글 드라이브와 코랩 연결 계정이 같은지 확인하세요.
- 방금 만든 폴더라면 Drive 동기화가 완료되었는지 확인하고 코랩 파일 목록의 새로고침을 누르세요.
- **공유 문서함**이나 **공유 드라이브**에만 있는 폴더는 내 드라이브와 위치가 다릅니다. 경로를 추측해서 입력하지 마세요. 이 경로 입력 기능은 `MyDrive` 안에 실제로 보이는 위치를 대상으로 합니다.
- 폴더 이름의 공백, 밑줄, 괄호도 일치해야 합니다. 직접 타이핑하기보다 경로 복사를 사용하세요.
- `연수자료` 같은 예시 폴더는 실제로 있을 때만 경로에 포함하세요.

## 이전 PKEMS 노트북을 계속 쓰는 경우
이전 버전은 전체 경로를 붙이면 `/content/drive/MyDrive/content/drive/MyDrive/...`처럼 중복됩니다. 이전 버전에서는 앞의 `/content/drive/MyDrive/`를 지우고 상대 경로만 넣으세요. 위의 전체 경로 자동 처리는 새 **PKOS_변환기.ipynb**에서 동작합니다. 기존에 Drive로 복사해둔 노트북은 자동으로 업데이트되지 않습니다.

1부는 블로그 PDF, 2부는 한글·워드·PDF 등 일반 파일, 3부는 구글 문서·시트·슬라이드용입니다. 2부의 문서폴더는 링크나 ID 입력을 지원하지 않습니다.


In [ ]:
#@title ▶ 폴더 훑어보기 (변환 없이 현황만) { display-mode: "form" }
#@markdown ### 변환할 폴더 (코랩에서 경로 복사 후 그대로 붙여넣기)
문서폴더 = "01_학교"  #@param {type:"string"}
#@markdown ### 결과를 저장할 폴더
결과폴더 = "PKOS/변환결과"  #@param {type:"string"}

import os
SRC_DIR = drive_path(문서폴더)
DST_DIR = drive_path(결과폴더)

if not os.path.isdir(SRC_DIR):
    print(f"⚠ 폴더를 찾을 수 없습니다: {SRC_DIR}")
else:
    fc = FolderConverter(FolderSettings(src_dir=SRC_DIR, out_dir=DST_DIR))
    fc.scan()

## 2-② 개인정보를 어떻게 가릴지 정하기

종류마다 처리 방식을 고를 수 있습니다.

| 방식 | 뜻 | 예시 |
|------|-----|------|
| **부분가림** | 일부만 남김 | 이운희 → `이**` · 010-1234-5678 → `010-****-****` |
| **가림** | 전부 가림 | 900101-1234567 → `******-*******` |
| **삭제** | 아예 지움 | (빈칸) |
| **그대로** | 건드리지 않음 | 이운희 |

In [ ]:
#@title ▶ 개인정보 설정 { display-mode: "form" }
#@markdown ### 개인정보를 가릴까요?
개인정보_가리기 = True  #@param {type:"boolean"}
#@markdown ---
#@markdown ### 종류별 처리 방식
주민등록번호 = "가림"      #@param ["가림", "부분가림", "삭제", "그대로"]
전화번호 = "부분가림"      #@param ["부분가림", "가림", "삭제", "그대로"]
이름 = "부분가림"          #@param ["부분가림", "가림", "삭제", "그대로"]
계좌번호 = "가림"          #@param ["가림", "부분가림", "삭제", "그대로"]
카드번호 = "가림"          #@param ["가림", "부분가림", "삭제", "그대로"]
이메일 = "부분가림"        #@param ["부분가림", "가림", "삭제", "그대로"]
주소 = "부분가림"          #@param ["부분가림", "가림", "삭제", "그대로"]
생년월일 = "부분가림"      #@param ["부분가림", "가림", "삭제", "그대로"]
차량번호 = "부분가림"      #@param ["부분가림", "가림", "삭제", "그대로"]
#@markdown ---
#@markdown ### 이름 찾는 강도
#@markdown `라벨만`=성명·담당자 옆의 이름만 · `보통`=+문서 안 반복 등장(권장) · `적극적`=+성씨 추정(오탐 늘어남)
이름_탐지강도 = "보통"  #@param ["보통", "라벨만", "적극적"]
#@markdown ### 보고서에 가리기 전 원본을 남길까요?
#@markdown 켜면 무엇이 바뀌었는지 대조할 수 있지만, **보고서 자체가 개인정보 덩어리**가 됩니다.
보고서_원본표시 = True  #@param {type:"boolean"}

정책 = Policy(
    주민등록번호=주민등록번호, 전화번호=전화번호, 이름=이름,
    계좌번호=계좌번호, 카드번호=카드번호, 이메일=이메일,
    주소=주소, 생년월일=생년월일, 차량번호=차량번호,
    이름_탐지강도=이름_탐지강도,
)
print("설정 완료")
for k, v in vars(정책).items():
    print(f"   {k:12} {v}")

### 미리보기 — 파일 하나로 시험해보기 (권장)

전체를 돌리기 전에 **파일 한 개**로 어떻게 가려지는지 확인해보세요.

In [ ]:
#@title ▶ 파일 하나로 미리보기 { display-mode: "form" }
#@markdown ### 확인할 파일 (내 드라이브 안 경로, 비우면 폴더에서 자동 선택)
확인할_파일 = ""  #@param {type:"string"}

import os

경로 = drive_path(확인할_파일) \
       if 확인할_파일.strip() else None

if 경로 is None:
    fc0 = FolderConverter(FolderSettings(src_dir=SRC_DIR, out_dir=DST_DIR))
    후보 = fc0.collect()
    경로 = 후보[0] if 후보 else None

if not 경로:
    print("확인할 파일을 찾지 못했습니다.")
else:
    print("파일 :", os.path.basename(경로), "\n")
    _r = read_any(경로)
    if not _r.ok:
        print("읽기 실패:", _r.error)
    else:
        개인정보_미리보기(_r.text, 정책)

## 2-③ 폴더 변환 시작

In [ ]:
#@title ▶ 폴더 변환 시작 { display-mode: "form" }
#@markdown ### 먼저 몇 개만 시험해볼까요? (0 = 전부)
시험_개수 = 30  #@param {type:"integer"}
#@markdown ### 이미 변환한 파일은 건너뛸까요?
중복_건너뛰기 = True  #@param {type:"boolean"}
#@markdown ### 원본 폴더 구조를 유지할까요?
폴더구조_유지 = True  #@param {type:"boolean"}

fc = FolderConverter(FolderSettings(
    src_dir        = SRC_DIR,
    out_dir        = DST_DIR,
    skip_existing  = 중복_건너뛰기,
    keep_tree      = 폴더구조_유지,
    개인정보_가리기 = 개인정보_가리기,
    개인정보_정책   = 정책,
    보고서_원본표시 = 보고서_원본표시,
))
결과 = fc.run(limit=(시험_개수 or None))

print()
print("저장 위치   :", DST_DIR)
print("목차        :", os.path.join(DST_DIR, "INDEX.md"))
print("개인정보보고서:", os.path.join(DST_DIR, "_개인정보_보고서.md"))

---
---

# 📄 3부 · 구글 문서·시트·슬라이드 가져오기

구글 문서는 **내 컴퓨터에 실체가 없는 온라인 문서**라서, 파일로는 읽을 수 없습니다.
Drive API 로 **내보내기(export)** 해야 합니다. (구글 문서 → 마크다운, 시트 → CSV)

처음 실행하면 계정 접근 허용을 한 번 더 물어봅니다.

In [ ]:
#@title ▶ ① 구글 문서 목록 보기 { display-mode: "form" }
#@markdown ### 폴더 링크 또는 ID
구글_폴더 = ""  #@param {type:"string"}
#@markdown ### 하위 폴더까지 찾을까요?
하위폴더_포함 = True  #@param {type:"boolean"}

import importlib, pkos_gdrive
importlib.reload(pkos_gdrive)
from pkos_gdrive import GoogleDocs

if not 구글_폴더.strip():
    print("폴더 링크나 ID를 입력해주세요.")
else:
    gd = GoogleDocs()
    문서목록 = gd.list_folder(구글_폴더, recursive=하위폴더_포함)

In [ ]:
#@title ▶ ② 구글 문서 가져오기 { display-mode: "form" }
#@markdown ### 저장할 폴더 (내 드라이브 안 경로)
구글_저장폴더 = "PKOS/구글문서"  #@param {type:"string"}

import os
G_OUT = drive_path(구글_저장폴더)
결과 = gd.export_folder(구글_폴더, G_OUT, recursive=하위폴더_포함)
print()
print("저장 위치 :", G_OUT)

---

### 잘 안 될 때

| 증상 | 해결 |
|------|------|
| 폴더를 찾을 수 없다 | 왼쪽 📁 아이콘 → `drive/MyDrive` 에서 실제 폴더명 확인 |
| 글을 0편 발견 | 네이버 블로그 **인쇄 → PDF 저장** 방식의 백업인지 확인 |
| 중간에 멈춤 | 코랩 연결이 끊긴 것. 다시 ▶ 누르면 **이어서** 진행됩니다 |
| 사진이 너무 많다 | `사진_저장`을 끄고 다시 실행 |

### 다음 단계

변환된 `.md` 파일들은 그대로 **나만의 지식창고**가 됩니다.
Claude·ChatGPT 같은 AI에게 폴더째 물어보거나, 웹 뷰어로 만들어 검색할 수 있습니다.

*PKOS · 개인지식운영체계*